In [1]:
import json
import numpy as np
from pathlib import Path
from datetime import datetime as dt

---

# Managing Reproducible Random Number Experiments

## Part 1: Understanding Pseudorandom Number Generators

See also: [https://en.wikipedia.org/wiki/Pseudorandom_number_generator](https://en.wikipedia.org/wiki/Pseudorandom_number_generator)

In [2]:
# The seed determines the initial state of the PRNG
seed = 42
rng = np.random.default_rng(seed)

In [3]:
# We don't often need to use this directly...
rng.bit_generator.state

{'bit_generator': 'PCG64',
 'state': {'state': 274674114334540486603088602300644985544,
  'inc': 332724090758049132448979897138935081983},
 'has_uint32': 0,
 'uinteger': 0}

In [4]:
#...generate some random data...
nums = rng.integers(low=0, high=2, size=(3, 5))
nums

array([[0, 1, 1, 0, 0],
       [1, 0, 1, 0, 0],
       [1, 1, 1, 1, 1]])

In [5]:
# ...and note that the state of the rng is now different...
rng.bit_generator.state

{'bit_generator': 'PCG64',
 'state': {'state': 175865384792420015972094667196815174992,
  'inc': 332724090758049132448979897138935081983},
 'has_uint32': 1,
 'uinteger': 3376120483}

In [6]:
# ...so if we generate more random data, it is different this time:
nums = rng.integers(low=0, high=2, size=(3, 5))
nums

array([[1, 1, 0, 1, 0],
       [1, 0, 0, 1, 1],
       [1, 0, 1, 1, 0]])

In [7]:
# If we reset the seed, we get the same numbers again
rng = np.random.default_rng(seed)
nums = rng.integers(low=0, high=2, size=(3, 5))
nums

array([[0, 1, 1, 0, 0],
       [1, 0, 1, 0, 0],
       [1, 1, 1, 1, 1]])

### Key Points

- The `rng` object is **stateful**
- Every next number generated depends on the current state
- Identical state $\implies$ identical output
- Asking `rng` for random numbers does two things:
    - It gives you numbers
    - It updates its state

If you rerun the above cells several times in a random order you will have no way to reproduce your results.

---

## Part 2: Producing/Reproducing Identical "Random" Data

The pattern I personally prefer is:

- Create and store raw data in batches
- Each batch is generated from:
    - A unique random seed
    - A specified sample size
- Batches are not edited after creation
- Write a separate process that manages random seeds

I'm not saying you must do things like this, but it should be a decent starting point.

We will finish writing the code in class, discussing as we go.

In [23]:
PATH_DECKS = Path('data/decks/')
PATH_SEED_LOG = Path('data/seed.json')
SEED_BASE = 42

def make_decks(seed: int, 
               n_decks: int,
               n_cards: int = 10
              ) -> np.ndarray:
    '''
    This code does NOT generate correct representations of card decks.
    Specifically, it does not generate an equal number of 0s and 1s.
    '''
    rng = np.random.default_rng(seed)
    return rng.integers(low=0, high=2, size=(n_decks, n_cards))

def get_next_seed() -> int:
    '''
    Read the last seed used, increment by 1,
    and update seed.json.
    '''
    # make sure the parent directory exists
    PATH_SEED_LOG.parent.mkdir(parents=True, exist_ok=True)

    # determine the next seed
    if not PATH_SEED_LOG.exists():
        print(f'No seed log found, starting with {SEED_BASE}')
        seed = SEED_BASE
    else:
        with PATH_SEED_LOG.open('r') as f:
            seed_log = json.load(f)
        seed = seed_log['seed'] + 1

    # update the log
    seed_log = {
        'seed': seed,
        'seed_time': str(dt.now())
    }
    with PATH_SEED_LOG.open('w') as f:
        json.dump(seed_log, f)
    
    return seed

def save_decks(decks: np.ndarray, 
               seed: int
              ) -> Path:
    '''
    This doesn't actually save anything,
    it is just a demo of how I might construct
    the filename.
    '''
    PATH_DECKS.mkdir(parents=True, exist_ok=True)

    n_decks = decks.shape[0]
    n_cards = decks.shape[1]

    filename = PATH_DECKS / f'decks_{n_decks}x{n_cards}_seed_{seed}.something'
    print(f'I might save this file like: {filename}')
    return filename

In [20]:
# Test each function as we go...

# decks = make_decks(42, 100)
# decks.shape

seed = get_next_seed()
print(seed)

50


In [24]:
# Test our sample pipeline
deck_size = 52
num_decks = 100
num_batches = 10

for n in range(num_batches):
    seed = get_next_seed()
    decks = make_decks(seed, num_decks, deck_size)
    save_decks(decks, seed)

I might save this file like: data/decks/decks_100x52_seed_52.something
I might save this file like: data/decks/decks_100x52_seed_53.something
I might save this file like: data/decks/decks_100x52_seed_54.something
I might save this file like: data/decks/decks_100x52_seed_55.something
I might save this file like: data/decks/decks_100x52_seed_56.something
I might save this file like: data/decks/decks_100x52_seed_57.something
I might save this file like: data/decks/decks_100x52_seed_58.something
I might save this file like: data/decks/decks_100x52_seed_59.something
I might save this file like: data/decks/decks_100x52_seed_60.something
I might save this file like: data/decks/decks_100x52_seed_61.something


---

## Other Notes

This example is intentionally minimal. If you were to go this route:

- You might also consider keeping a separate log of which files were created and when, including the seed and array size. The idea is, if there were ever a conflict between the log and the contents of the files or filenames, you would have a clue that there might be a bug in the code.
- If the same seed is ever used twice, something might be wrong.
- Consider a minor bug in the logic:
    - `get_next_seed` currently updates the last seed used before any data is generated. If another part of the program were to crash, the power went out, etc., then it might look like we have used seeds that we didn't.
 
Ultimately, your goal should be to have a readily auditable and reproducible workflow. There's not neccesarily a single best way to do this, and as the project matures, you may realize that you need to rethink some of your previous decisions. This is normal!